# M4.6 · Attention ve multi-head attention

Scaled dot-product attention'ı 3 token'lık bir örnekte elle hesaplıyor, saf Python'da maskeli tek-head attention yazıyor, NumPy'da batch'li multi-head attention'ı döngüsüz kuruyor, PyTorch'ta `nn.MultiheadAttention` ile birebir aynı sonucu veren bir modül yazıyor ve en sonda FlashAttention'ın çekirdek fikri olan online softmax ile T×T skor matrisini hiç oluşturmadan attention hesaplıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-6-attention.html](https://m-gonul.github.io/pytorch-lab/m4-6-attention.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_6_attention.py
CHECKS = {
 "b": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef ref(Q, K, V, causal):\n    Q, K, V = map(np.array, (Q, K, V))\n    S = Q @ K.T / np.sqrt(Q.shape[1])\n    if causal:\n        S = np.where(np.triu(np.ones(S.shape), 1) > 0, -np.inf, S)\n    A = np.exp(S - S.max(1, keepdims=True)); A /= A.sum(1, keepdims=True)\n    return A @ V\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    R.test(\"softmax_row: büyük değer ve -inf\", lambda: close(need(ns, \"softmax_row\")([1000.0, 1000.0, float(\"-inf\")]), [0.5, 0.5, 0.0]))\n    Q = K = [[1, 0], [0, 1], [1, 1]]; V = [[1, 2], [3, 4], [5, 6]]\n    R.test(\"elle örnek\", lambda: close(need(ns, \"attention\")(Q, K, V), ref(Q, K, V, False)))\n    R.test(\"elle örnek, causal\", lambda: close(need(ns, \"attention\")(Q, K, V, causal=True), ref(Q, K, V, True)))\n\n    def rnd():\n        rng = np.random.default_rng(0)\n        Q, K, V = rng.normal(size=(5, 4)), rng.normal(size=(5, 4)), rng.normal(size=(5, 3))\n        for c in (False, True):\n            close(need(ns, \"attention\")(Q.tolist(), K.tolist(), V.tolist(), causal=c), ref(Q, K, V, c), what=f\"causal={c}\")\n    R.test(\"rastgele 5×4, d_v = 3\", rnd)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef ref_mha(x, Wq, Wk, Wv, Wo, h, causal):\n    B, T, d = x.shape; dk = d // h\n    out = np.zeros((B, T, d))\n    for b in range(B):\n        heads = []\n        for i in range(h):\n            sl = slice(i * dk, (i + 1) * dk)\n            q, k, v = x[b] @ Wq[:, sl], x[b] @ Wk[:, sl], x[b] @ Wv[:, sl]\n            S = q @ k.T / np.sqrt(dk)\n            if causal:\n                S[np.triu_indices(T, 1)] = -np.inf\n            A = np.exp(S - S.max(1, keepdims=True)); A /= A.sum(1, keepdims=True)\n            heads.append(A @ v)\n        out[b] = np.concatenate(heads, 1) @ Wo\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    R.test(\"softmax: -inf ve büyük değer\", lambda: close(need(ns, \"softmax\")(np.array([[1000.0, float(\"-inf\"), 1000.0]])), [[0.5, 0, 0.5]]))\n    rng = np.random.default_rng(1)\n    x = rng.normal(size=(3, 6, 12)); W = [rng.normal(size=(12, 12)) / 3 for _ in range(4)]\n\n    def t(h, c):\n        return lambda: close(need(ns, \"mha\")(x, *W, h=h, causal=c), ref_mha(x, *W, h, c))\n    R.test(\"h = 3, tam\", t(3, False))\n    R.test(\"h = 3, causal\", t(3, True))\n    R.test(\"h = 1 (tek head)\", t(1, False))\n\n    def causal_prop():\n        y = need(ns, \"mha\")(x, *W, h=4, causal=True)\n        x2 = x.copy(); x2[:, 4:] = rng.normal(size=x2[:, 4:].shape)\n        y2 = need(ns, \"mha\")(x2, *W, h=4, causal=True)\n        close(y[:, :4], y2[:, :4], what=\"ilk 4 konum\")\n        ensure(not np.allclose(y[:, 4:], y2[:, 4:]), \"son konumlar değişmeliydi\")\n    R.test(\"causal: gelecek değişince geçmiş çıktılar değişmiyor\", causal_prop)\n",
 "d": "import ast\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n\n    def no_sdpa():\n        for node in ast.walk(ast.parse(src)):\n            if isinstance(node, ast.FunctionDef) and node.name == \"attention\":\n                names = {n.attr for n in ast.walk(node) if isinstance(n, ast.Attribute)} | {n.id for n in ast.walk(node) if isinstance(n, ast.Name)}\n                ensure(\"scaled_dot_product_attention\" not in names, \"`attention` içinde F.scaled_dot_product_attention kullanma; elle yaz\")\n                return\n        raise AssertionError(\"`attention` fonksiyonu yok\")\n    R.test(\"attention elle yazılmış\", no_sdpa)\n    g = torch.Generator().manual_seed(0)\n    q, k, v = (torch.randn(2, 3, 9, 8, generator=g, dtype=torch.float64) for _ in range(3))\n    R.test(\"attention = SDPA\", lambda: close(need(ns, \"attention\")(q, k, v), F.scaled_dot_product_attention(q, k, v)))\n    R.test(\"attention = SDPA (is_causal)\", lambda: close(need(ns, \"attention\")(q, k, v, causal=True), F.scaled_dot_product_attention(q, k, v, is_causal=True)))\n\n    def vs_torch(causal):\n        def f():\n            torch.manual_seed(0)\n            m = need(ns, \"MultiHeadAttention\", \"sınıf\")(16, 4).double()\n            ensure(isinstance(getattr(m, \"qkv\", None), nn.Linear) and isinstance(getattr(m, \"proj\", None), nn.Linear), \"self.qkv ve self.proj nn.Linear olmalı\")\n            ref = nn.MultiheadAttention(16, 4, batch_first=True).double()\n            with torch.no_grad():\n                ref.in_proj_weight.copy_(m.qkv.weight); ref.in_proj_bias.copy_(m.qkv.bias)\n                ref.out_proj.weight.copy_(m.proj.weight); ref.out_proj.bias.copy_(m.proj.bias)\n            x = torch.randn(3, 7, 16, generator=g, dtype=torch.float64)\n            mask = torch.ones(7, 7, dtype=torch.bool).triu(1) if causal else None\n            want, _ = ref(x, x, x, attn_mask=mask, need_weights=False)\n            close(m(x, causal=causal), want)\n        return f\n    R.test(\"nn.MultiheadAttention ile aynı\", vs_torch(False))\n    R.test(\"nn.MultiheadAttention ile aynı (causal)\", vs_torch(True))\n\n    def grad():\n        torch.manual_seed(1)\n        m = need(ns, \"MultiHeadAttention\", \"sınıf\")(8, 2)\n        m(torch.randn(2, 5, 8), causal=True).sum().backward()\n        ensure(all(p.grad is not None for p in m.parameters()), \"tüm parametreler gradyan almalı\")\n    R.test(\"geri geçiş\", grad)\n",
 "e": "import math\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"blocked_attention'da tek döngü (bloklar)\", lambda: max_loops(src, 1, func=\"blocked_attention\", why=\"satır ya da eleman döngüsü yok\"))\n\n    def ast_no_sdpa():\n        import ast\n        for node in ast.walk(ast.parse(src)):\n            if isinstance(node, ast.FunctionDef) and node.name == \"blocked_attention\":\n                names = {n.attr for n in ast.walk(node) if isinstance(n, ast.Attribute)}\n                ensure(\"scaled_dot_product_attention\" not in names and \"softmax\" not in names,\n                       \"SDPA ya da softmax çağırma; online softmax'ı elle yaz\")\n                return\n        raise AssertionError(\"`blocked_attention` yok\")\n    R.test(\"SDPA/softmax çağrısı yok\", ast_no_sdpa)\n    g = torch.Generator().manual_seed(0)\n    q, k, v = (torch.randn(2, 3, 150, 16, generator=g, dtype=torch.float64) for _ in range(3))\n\n    def eq(block, causal):\n        return lambda: close(need(ns, \"blocked_attention\")(q, k, v, block=block, causal=causal),\n                             F.scaled_dot_product_attention(q, k, v, is_causal=causal))\n    R.test(\"SDPA ile aynı (blok 64, 150 bölünmüyor)\", eq(64, False))\n    R.test(\"SDPA ile aynı (causal, blok 32)\", eq(32, True))\n    R.test(\"blok 1 (en uç durum, causal)\", eq(1, True))\n\n    def big():\n        qq = q * 30\n        close(need(ns, \"blocked_attention\")(qq, k, v, block=40), F.scaled_dot_product_attention(qq, k, v), what=\"büyük skorlar\")\n    R.test(\"büyük skorlarda taşma yok\", big)\n\n    def grad():\n        a = [t.clone().requires_grad_(True) for t in (q, k, v)]\n        b = [t.clone().requires_grad_(True) for t in (q, k, v)]\n        need(ns, \"blocked_attention\")(*a, block=48, causal=True).pow(2).sum().backward()\n        F.scaled_dot_product_attention(*b, is_causal=True).pow(2).sum().backward()\n        for x, y, n in zip(a, b, \"qkv\"):\n            close(x.grad, y.grad, what=f\"d{n}\")\n    R.test(\"gradyanlar SDPA ile aynı\", grad)\n    if \"blocked_attention\" not in ns:\n        return\n    T = 2048\n    Q, K, V = (torch.randn(1, 8, T, 64, generator=g) for _ in range(3))\n\n    def naive():\n        return ((Q @ K.transpose(-2, -1)) / 8).softmax(-1) @ V\n    t_n = bench(naive, repeat=3)\n    t_b = bench(lambda: ns[\"blocked_attention\"](Q, K, V, block=256), repeat=3)\n    t_s = bench(lambda: F.scaled_dot_product_attention(Q, K, V), repeat=3)\n    mb = lambda n: f\"{n * 4 / 2**20:.0f} MB\"\n    R.table(f\"T = {T}, 8 head, d_k = 64\", [\"yöntem\", \"süre\", \"skor ara belleği\"],\n            [[\"naif (tam T×T)\", fmt_time(t_n), mb(8 * T * T)],\n             [\"senin blocked (blok 256)\", fmt_time(t_b), mb(8 * T * 256)],\n             [\"F.scaled_dot_product_attention\", fmt_time(t_s), \"fused\"]],\n            note=\"Bloklu yol CPU'da bile hızlanabilir: blok önbelleğe sığar. GPU'da aynı fikir blokları SRAM'de tutup HBM trafiğini keser (FlashAttention).\")\n"
}

In [ ]:
import importlib, labcheck, checks_m4_6_attention as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Her token için üç vektör: **query** $q$ (ne arıyorum), **key** $k$ (neyim), **value** $v$ (ne veririm). Scaled dot-product attention:

$$
\operatorname{Attn}(Q,K,V)=\operatorname{softmax}\!\Big(\frac{QK^\top}{\sqrt{d_k}}+M\Big)V,
$$

$M_{ij}=-\infty$ ise $i$, $j$'yi görmez (causal maske: $j>i$). Çıktının $i$. satırı, $V$ satırlarının $A_{i,:}$ ağırlıklarıyla ortalaması.

**Multi-head:** $d$ boyutunu $h$ parçaya böl ($d_k=d/h$), her head kendi attention'ını yapsın, sonuçlar birleştirilip $W_O$ ile karıştırılsın. Kodda bu, `(B, T, d)` → `(B, h, T, d_k)` reshape'i ve tek bir batch'li matmul.

**Online softmax:** $\operatorname{softmax}$'ı bloklar hâlinde, satır başına koşan bir maksimum $m$ ve payda $\ell$ tutarak hesaplayabilirsin. Yeni blokla $m'=\max(m,\max_j s_j)$:

$$
\ell'=\ell\,e^{m-m'}+\sum_j e^{s_j-m'},\qquad o'=o\,e^{m-m'}+\sum_j e^{s_j-m'}v_j,\qquad \text{sonuç}=o/\ell.
$$

## 1 · Elle

1. satırda $q_1=[1,0]$; skorlar $[1,0,1]/\sqrt2$. 1. ve 3. key'e eşit ağırlık düşüyor, bu yüzden çıktı $v_1$ ile $v_3$'ün ortasına, $[3,4]$'e çekiliyor ($v_2$'nin payı da tam ortada olduğu için ortalamayı bozmuyor).

Causal'da 2. sorgu yalnız $k_1, k_2$'yi görüyor: $\operatorname{softmax}([0, 0.7071])$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$Q=K=\begin{bmatrix}1&0\\0&1\\1&1\end{bmatrix}$, $V=\begin{bmatrix}1&2\\3&4\\5&6\end{bmatrix}$, $d_k=2$. Skor $S=QK^\top/\sqrt2$**

- `s13`: $S_{1,3}$
- `s12`: $S_{1,2}$
- `w11`: 1. satır softmax: $A_{1,1}$
- `o11`: Çıktı $(AV)_{1,1}$

**Causal maske: $j>i$ için $S_{ij}=-\infty$**

- `c11`: Causal çıktı $(AV)_{1,1}$
- `c22w`: Causal $A_{2,2}$
- `c21`: Causal çıktı $(AV)_{2,1}$

In [ ]:
cevaplar = dict(
    s13=None,
    s12=None,
    w11=None,
    o11=None,
    c11=None,
    c22w=None,
    c21=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"s13":[0.707107,0.0005],"s12":[0,0],"w11":[0.401112,0.0005],"o11":[3,0.0005],"c11":[1,0],"c22w":[0.669762,0.0005],"c21":[2.339524,0.001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, liste listeleri:

- `softmax_row(xs)` — `-inf` girdileri destekle (max'ı çıkar; `math.exp(-inf) = 0`).
- `attention(Q, K, V, causal=False)` → `(T, d_v)` liste; $T\times d_k$ girdiler, ölçek $1/\sqrt{d_k}$.

In [ ]:
%%writefile stage_b.py
import math


def softmax_row(xs):
    ...


def attention(Q, K, V, causal=False):
    ...


Q = K = [[1, 0], [0, 1], [1, 1]]
V = [[1, 2], [3, 4], [5, 6]]
print(attention(Q, K, V))               # 1. satır [3, 4]
print(attention(Q, K, V, causal=True))  # 1. satır [1, 2]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**. `x: (B, T, d)`, ağırlıklar `Wq, Wk, Wv, Wo: (d, d)` (çarpım `x @ W`).

- `softmax(x, axis=-1)` — kararlı, `-inf` destekli.
- `mha(x, Wq, Wk, Wv, Wo, h, causal=False)` → `(B, T, d)`.

Yol: `q = x @ Wq` → `(B, T, h, dk)` → `transpose(0, 2, 1, 3)` → `(B, h, T, dk)`; skorlar `q @ k.transpose(0, 1, 3, 2)`; maske `np.triu(…, 1)`; sonuç geri `(B, T, d)` ve `@ Wo`.

In [ ]:
%%writefile stage_c.py
import numpy as np


def softmax(x, axis=-1):
    ...


def mha(x, Wq, Wk, Wv, Wo, h, causal=False):
    ...


rng = np.random.default_rng(0)
x = rng.normal(size=(2, 5, 8)); W = rng.normal(size=(4, 8, 8)) / 3
print(mha(x, *W, h=2, causal=True).shape)   # (2, 5, 8)

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `attention(q, k, v, causal=False)` — `(…, T, dk)` girdiler; **elle** (`F.scaled_dot_product_attention` çağırmadan). Kontrol onunla karşılaştırır.
- `MultiHeadAttention(d, h)` — `nn.Module`:
  - `self.qkv = nn.Linear(d, 3 * d)` (çıktı sırası q, k, v), `self.proj = nn.Linear(d, d)`.
  - `forward(x, causal=False)` — `(B, T, d)` → `(B, T, d)`. Head'ler: `(B, T, h, dk)` → `transpose(1, 2)`.

Bu yerleşim `nn.MultiheadAttention(batch_first=True)`'ın `in_proj_weight`/`out_proj`'u ile birebir aynı; kontrol ağırlıkları kopyalayıp çıktıları karşılaştırır.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn.functional as F
from torch import nn


def attention(q, k, v, causal=False):
    """(…, T, dk). F.scaled_dot_product_attention kullanmadan."""
    ...


class MultiHeadAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        ...

    def forward(self, x, causal=False):
        ...


torch.manual_seed(0)
m = MultiHeadAttention(16, 4)
print(m(torch.randn(2, 7, 16), causal=True).shape)   # (2, 7, 16)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`blocked_attention(q, k, v, block=64, causal=False)` — key/value'ları `block` uzunluğunda parçalar hâlinde gez ve online softmax ile birleştir. **Yalnız blok döngüsü** (tek katman). Satır başına `m` (koşan max, başlangıç $-\infty$), `l` (payda), `acc` (pay).

Dikkat: causal'da bir satırın bir bloktaki tüm skorları $-\infty$ olabilir, hatta o ana kadarki `m` de $-\infty$ olabilir; $-\infty-(-\infty)$ `nan` verir. `m_new` $-\infty$ ise çıkarılacak değer olarak 0 kullan.

Kontrol sonucu ve gradyanı `F.scaled_dot_product_attention` ile karşılaştırır. Asıl kazanç bellekte: tam skor matrisi yerine `T × block`. CPU'da da blok önbelleğe sığdığı için genelde hızlanır; GPU'da bu fikir SRAM/HBM farkından büyük hız getirir.

In [ ]:
%%writefile stage_e.py
import math
import torch
import torch.nn.functional as F


def blocked_attention(q, k, v, block=64, causal=False):
    """(…, T, dk). Tek döngü: key/value blokları. T×T skor matrisi yok."""
    ...


g = torch.Generator().manual_seed(0)
q, k, v = (torch.randn(1, 2, 200, 16, generator=g) for _ in range(3))
print((blocked_attention(q, k, v, 64, causal=True) - F.scaled_dot_product_attention(q, k, v, is_causal=True)).abs().max())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Dao ve ark. (2022): FlashAttention

[Dao, Fu, Ermon, Rudra, Ré — FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness (NeurIPS 2022)](https://arxiv.org/abs/2205.14135)

e aşamasındaki online softmax'ı bir Triton çekirdeğine (M7.2) çevir: Q bloğu başına K/V bloklarını SRAM'e yükle, satır maksimumu ve paydayı güncelle (Algoritma 1), çıktıyı yeniden ölçekle. Geri geçişi makalenin yeniden hesaplama yaklaşımıyla (Algoritma 2; yalnız logsumexp sakla) yaz. Bellek kullanımını (O(N) vs O(N²)) ve HBM okuma sayısını kuramsal formülle (Teorem 2) karşılaştır; nedensel maske ve blok atlama ekle.


**Adımlar**

1. GPU yoksa TRITON_INTERPRET=1 ile doğruluğu, Colab'da hızı ölç.

**Kabul ölçütleri**

- Çıktı ve gradyanlar standart attention'la 10⁻³ içinde.
- Uzun dizide bellek ve süre kazancı tablosu.

## Kaynaklar

- [Vaswani ve ark. — Attention Is All You Need](https://arxiv.org/abs/1706.03762) — arXiv
- [Karpathy — nanoGPT / Zero to Hero: Let's build GPT](https://github.com/karpathy/nanoGPT) — MIT
- [Dao ve ark. — FlashAttention](https://arxiv.org/abs/2205.14135) — arXiv
- [Milakov & Gimelshein — Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867) — arXiv
- [UvA Deep Learning Tutorials — Transformers and Multi-Head Attention](https://uvadlc-notebooks.readthedocs.io/) — MIT
- [Stanford CS336 — Assignment 1/2 (attention, FlashAttention-2 Triton)](https://github.com/stanford-cs336) — MIT